In [1]:
import json
import pennylane as qp
import pennylane.numpy as np

In [ ]:
def wrapper(n: int) -> float:
    """Builds the n-qubit GHZ state and averages its n parity checks.

    Args:
        n (int): The number of qubits. Guaranteed to satisfy n >= 2.

    Returns:
        float: The mean of the n parity values -- the global X parity together with
            the n-1 neighbouring ZZ parities. This equals +1 if and only if the
            state is the GHZ state.
    """
    dev = qp.device("default.qubit", wires=n)

    @qp.qnode(dev)
    def parities():
        return [qp.expval(observable) for observable in prepare_and_build_checks(n)]

    return float(np.mean(np.array(parities())))


def prepare_and_build_checks(n):
    """Prepares the GHZ state and lists the parity checks to measure on it.

    This runs inside the QNode above, so any gate you apply here is applied to the
    circuit, and the observables you return are the ones that get measured.

    Args:
        n (int): the number of qubits, guaranteed to be at least 2.

    Returns:
        list[Operator]: the n observables to measure -- the global X parity and the
            n-1 neighbouring ZZ parities. The order does not matter.
    """
    # Step 1: build the GHZ state -- a Hadamard on wire 0, then a CNOT fan-out
    #         from wire 0 to every other wire.
    # Step 2: build the global X parity: an X on every wire, multiplied together
    #         into a single observable with qp.prod and assign it 'checks'.
    # Step 3: build the n-1 neighbouring ZZ parities, one for each adjacent pair,
    #         indexed by the left-hand qubit of the pair and add it to 'checks'.
    # Step 4: collect all n observables into checks and return it.

    checks = []

    # Step 1:
    qp.Hadamard(wires=0)
    for i in range(n):
        qp.CNOT(wires=[0, i+1])
    
    # Step 2:
    x_parity = qp.prod()
        
    checks.append(x_parity)

    # Step 3:
    for i in range(n-1):
        obs = qp.PauliZ(wires=i) @ qp.PauliZ(wires=i+1)

        checks.append(obs)
    
    return checks